# Notebook 01 — Temporal Integration Sweep
### Memory-Augmented Brain Encoding (rigorous rebuild) · `Mrabbi3/memory-augmented-brain-encoding`

**The question.** Where in cortex does integrating stimulus information over *longer* timescales improve our ability to predict brain activity? This is the **temporal receptive window (TRW) hierarchy**: early sensory cortex processes moment-to-moment; higher-order association cortex (default-mode, frontoparietal control) accumulates information over longer stretches.

**The design (why it's clean).** We hold model capacity fixed — the same 250 PCA features, the same ridge regression — and vary **only** the temporal context length `L`: to predict the fMRI at TR *t*, the model sees the causal mean of features over the past `L` TRs. Because nothing but the integration window changes, any difference in prediction is attributable to temporal integration alone, not model size. This directly addresses the reviewer feedback (reframe around temporal integration / recurrence) and avoids the v1 problems (no circular features, no per-TR strawman — `L=1` is just one point on a principled curve).

**Pipeline:** load aligned data (all available subjects) → split by **episode** (no temporal leakage) → fit a shared PCA on training features → sweep `L` ∈ {1,2,4,8,16,32,64,100} → per-parcel test correlation → group parcels by network and look at the curves. Raw correlations here; **noise-ceiling normalization and the formal cortical hierarchy map come in NB02.**

> Prereq: run NB00 with `RUN_ALL_SUBJECTS = True` so all four `*_friends_aligned.npz` files exist. This notebook works with 1+ subjects but is more robust with 4.

## Config

In [ ]:
import numpy as np, glob, os

DRIVE_ROOT  = "/content/drive/MyDrive/Research/memory-brain-encoding"
ALIGNED_DIR = f"{DRIVE_ROOT}/data/aligned"
RESULTS_DIR = f"{DRIVE_ROOT}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CONTEXT_LENGTHS = [1, 2, 4, 8, 16, 32, 64, 100]   # TRs of causal context (1 TR = 1.49 s)
N_PCA           = 250                              # feature dim held FIXED across all L
RIDGE_ALPHAS    = np.logspace(2, 6, 9)
TEST_FRAC       = 0.2                              # fraction of EPISODES held out
SEED            = 0

subject_files = sorted(glob.glob(ALIGNED_DIR + "/*_friends_aligned.npz"))
print(f"Found {len(subject_files)} subject file(s):")
for f in subject_files: print("  ", os.path.basename(f))
assert subject_files, "No aligned .npz found — run NB00 Step 5 first."

## Mount Drive & install (nilearn gives us the Schaefer network labels)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q nilearn scikit-learn
print("ready")

## Helpers
- **Episode split** — held-out episodes are chosen once and applied to every subject, so train/test are temporally disjoint (no TR from a test episode ever leaks into training).
- **Causal pooling** — the mean of features over the past `L` TRs, computed *within* each episode (never across episode boundaries) via a cumulative-sum trick.
- **Per-parcel correlation** — vectorized Pearson r between predicted and true fMRI, one value per parcel.

In [ ]:
def make_split(episodes, test_frac=TEST_FRAC, seed=SEED):
    uniq = sorted(set(episodes.tolist()))
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(uniq))
    n_test = max(1, int(round(len(uniq) * test_frac)))
    return set(uniq[i] for i in perm[:n_test])

def tr_masks(episodes, bounds, test_ids):
    T = int(bounds[-1])
    test = np.zeros(T, dtype=bool)
    for i, ep in enumerate(episodes):
        if ep in test_ids:
            test[int(bounds[i]):int(bounds[i+1])] = True
    return ~test, test

def causal_pool(Xp, bounds, L):
    # Causal mean over the past L TRs, reset at every episode boundary.
    if L == 1:
        return Xp
    out = np.empty_like(Xp)
    for i in range(len(bounds) - 1):
        a, b = int(bounds[i]), int(bounds[i+1])
        seg = Xp[a:b]
        cs = np.cumsum(seg, axis=0)
        pooled = cs.copy()
        pooled[L:] = cs[L:] - cs[:-L]
        counts = np.minimum(np.arange(1, len(seg) + 1), L)[:, None]
        out[a:b] = pooled / counts
    return out

def col_corr(pred, true):
    p = pred - pred.mean(0); t = true - true.mean(0)
    num = (p * t).sum(0)
    den = np.sqrt((p**2).sum(0) * (t**2).sum(0)) + 1e-8
    return num / den
print("helpers defined")

## Fit a shared PCA on the training features
The stimulus features are identical across subjects, so we fit one feature basis (on the *training* TRs only, to avoid leakage) and reuse it for everyone. This keeps the experiment comparable across subjects and folds.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

d0 = np.load(subject_files[0])
test_ids = make_split(d0["episodes"])
print(f"held-out episodes: {len(test_ids)} (of {len(set(d0['episodes'].tolist()))})")

X0, b0, e0 = d0["X"], d0["bounds"], d0["episodes"]
trm0, tem0 = tr_masks(e0, b0, test_ids)
print(f"train TRs: {trm0.sum()},  test TRs: {tem0.sum()}")

scaler = StandardScaler().fit(X0[trm0])
pca = PCA(n_components=N_PCA, svd_solver="randomized", random_state=SEED).fit(scaler.transform(X0[trm0]))
print(f"PCA: {N_PCA} comps explain {pca.explained_variance_ratio_.sum()*100:.1f}% of feature variance")
del X0

## The context-length sweep
For each subject and each `L`, we pool features over the past `L` TRs, fit ridge (alpha chosen per parcel by fast generalized cross-validation), and record the test correlation per parcel. `all_r` ends up shaped **(subjects, L, 1000 parcels)**.

In [ ]:
from sklearn.linear_model import RidgeCV
import time

n_sub = len(subject_files)
all_r = np.zeros((n_sub, len(CONTEXT_LENGTHS), 1000), dtype=np.float32)
subjects = []

for si, path in enumerate(subject_files):
    sub = os.path.basename(path).split("_")[0]; subjects.append(sub)
    d = np.load(path)
    X, y, bounds, eps = d["X"], d["y"], d["bounds"], d["episodes"]
    trm, tem = tr_masks(eps, bounds, test_ids)
    Xp = pca.transform(scaler.transform(X))
    t0 = time.time()
    for li, L in enumerate(CONTEXT_LENGTHS):
        Xpool = causal_pool(Xp, bounds, L)
        m = RidgeCV(alphas=RIDGE_ALPHAS, alpha_per_target=True)
        m.fit(Xpool[trm], y[trm])
        all_r[si, li] = col_corr(m.predict(Xpool[tem]), y[tem])
    print(f"{sub}: swept {len(CONTEXT_LENGTHS)} context lengths in {time.time()-t0:.0f}s  "
          f"(mean r @L=1: {all_r[si,0].mean():.3f}, best-L mean r: {all_r[si].max(0).mean():.3f})")
    del X, y, Xp

print("\nsweep done. all_r shape:", all_r.shape)

## Group parcels by network (Schaefer 1000 / 7 networks)
We fetch the standard Schaefer-2018 1000-parcel, 7-network labels and assign each parcel to one of: Visual, SomatoMotor, DorsalAttention, SalienceVentralAttention, Limbic, Control (frontoparietal), Default (DMN). The TRW hypothesis predicts the optimal context length should grow as we move from sensory (Vis/SomMot) toward association cortex (Control/Default).

In [ ]:
from nilearn import datasets

atlas = datasets.fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=7, resolution_mm=2)
labels = [l.decode() if isinstance(l, bytes) else l for l in atlas["labels"]]
NETS = ["Vis", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Cont", "Default"]
def net_of(lbl):
    for n in NETS:
        if n in lbl: return n
    return "Other"
networks = np.array([net_of(l) for l in labels])
assert len(networks) == 1000, f"expected 1000 labels, got {len(networks)}"
for n in NETS:
    print(f"  {n:12s}: {(networks==n).sum()} parcels")

## Result: does longer context help, and where?
Mean test correlation vs. context length, one curve per network (averaged across subjects and the parcels in each network). If the TRW hypothesis holds, sensory networks should peak at short `L` and flatten/decline, while association networks should keep benefiting from longer `L`.

In [ ]:
import matplotlib.pyplot as plt

r_mean = all_r.mean(0)   # (L, 1000) averaged over subjects
xs = np.array(CONTEXT_LENGTHS)

plt.figure(figsize=(9, 5.5))
for n in NETS:
    idx = networks == n
    if idx.sum() == 0: continue
    plt.plot(xs, r_mean[:, idx].mean(1), marker="o", label=f"{n} ({idx.sum()})")
plt.xscale("log", base=2); plt.xticks(xs, xs)
plt.xlabel("context length L (TRs, 1 TR = 1.49 s)")
plt.ylabel("mean test correlation r")
plt.title("Encoding accuracy vs. temporal context, by network")
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Optimal context length per parcel, summarized by network
opt_L = np.array(CONTEXT_LENGTHS)[r_mean.argmax(0)]   # (1000,)
print("Median optimal context length (TRs) per network:")
order = sorted(NETS, key=lambda n: np.median(opt_L[networks==n]) if (networks==n).any() else 0)
for n in order:
    idx = networks == n
    if idx.sum() == 0: continue
    print(f"  {n:12s}: median L = {int(np.median(opt_L[idx])):3d}   "
          f"(mean best r = {r_mean[:, idx].max(0).mean():.3f})")

## Save results for NB02

In [ ]:
out = os.path.join(RESULTS_DIR, "sweep_r.npz")
np.savez_compressed(out,
    all_r=all_r, context_lengths=np.array(CONTEXT_LENGTHS),
    subjects=np.array(subjects), networks=networks,
    test_ids=np.array(sorted(test_ids)), n_pca=N_PCA)
print("saved", out, "| all_r", all_r.shape)

## What this is, and what's next
This is a real, leakage-controlled encoding result: the test episodes were never seen in training, capacity is fixed across `L`, and the features are genuine multimodal stimulus representations. Raw correlations look modest (that's normal for fMRI encoding) — **NB02** divides them by a per-parcel **noise ceiling** (inter-subject correlation) so we can report *fraction of explainable variance*, then makes the formal cortical map and runs significance tests across subjects/episodes.

The bigger arc: this notebook tests **fixed-window** integration. If the network curves show association cortex benefiting from longer `L`, the natural next question — and where your memory module earns its place — is whether **learned, content-based** integration (recurrence / attention over history) beats a fixed window, especially in high-order regions. That's NB03.